# Tutorial 04: Spectral Extraction

This tutorial turns a 2-D frame into a row-stacked spectrum. It covers both
extraction methods in the pipeline, and measures what separates them.

**Two methods:**

- **Boxcar** sums the pixels inside an aperture around each trace. Fast, hard
  to break, and the right default.
- **Optimal** (flat-relative, FOX) weights every pixel by the fiber profile
  taken from the flat, and by the noise.

**The one thing worth remembering:** the two agree closely at this instrument,
and the size of the difference is set by how much of the aperture is empty
pixels. Both statements are measured below rather than asserted.

**Before you start:** install the package and Jupyter in an environment that
has `numpy`, `astropy`, `matplotlib` and `scipy`; see `docs/CONTRIBUTING.md`.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from amasedrp.simulate import synthetic_fiber_flat, synthetic_arc
from amasedrp.reduction import (
    identify_and_trace_fibers,
    extract_spectra,
    run_quick_reduction,
    run_reduction,
)
from amasedrp.reduction.methods.boxcar import extract_boxcar
from amasedrp.reduction.methods.optimal import extract_optimal

## 1. A flat, a science frame, and a trace model

Extraction needs all three. The flat gives the fiber positions and the profile;
the trace model says where each fiber lies on every row; the science frame
carries the light to be measured.

The synthetic arc has three emission lines, so the extracted spectra have
structure to compare.

In [ ]:
n_rows = 1200
flat, true_centers = synthetic_fiber_flat(
    n_rows=n_rows, n_blocks=3, n_fibers_per_block=5,
)
science, wavelength = synthetic_arc(
    flat,
    lines=np.array([4657.9012, 4764.8646, 4965.0795]),
    wavelength_zero=4600.0,
    dispersion=0.35,
    line_fwhm=1.2,
    continuum=0.5,
    noise_std=0.01,
    seed=11,
)

fibermap, tracemask = identify_and_trace_fibers(
    image=flat,
    n_blocks_expected=3,
    n_fibers_per_block_expected=5,
    poly_deg=3,
)

print(f"fibers      : {fibermap.n_fibers}")
print(f"flat shape  : {flat.shape}")
print(f"science     : {science.shape}, noise 0.01 ADU per pixel")
print(f"aperture    : default radius 3, so 7 pixels per row")

## 2. Boxcar extraction

The aperture is centred on the trace at every row, and the pixels inside it are
summed. No model, no weights: whatever light falls in the aperture is counted,
including light from a neighbouring fiber that reaches into it.

In [ ]:
boxcar_frame = extract_spectra(
    image=science,
    tracemask=tracemask,
    fibermap=fibermap,
    method="boxcar",
    aperture_radius=3,
)

print(f"shape : {boxcar_frame.shape}")
print(f"meta  : {boxcar_frame.meta}")

fig, ax = plt.subplots(figsize=(11, 3.2))
for index in range(boxcar_frame.n_fibers):
    ax.plot(boxcar_frame.wave, boxcar_frame.flux[index], lw=0.7, alpha=0.85)
ax.set_xlabel("pixel (dispersion direction)")
ax.set_ylabel("flux [ADU]")
ax.set_title("Boxcar extraction, all 15 fibers")
plt.show()

## 3. Optimal extraction (FOX)

Flat-relative optimal extraction uses the flat as the model of the fiber
profile. Every pixel is weighted by the profile and by the noise, and the
weighted sum of the science is divided by the weighted sum of the flat. A
smooth version of the boxcar-extracted flat then removes the lamp spectrum and
returns the result to the same scale as a boxcar sum.

`method="optimal"` needs the flat, the gain and the read noise. It builds its
own noise model from the science counts, so passing a variance image is
rejected rather than silently ignored.

In [ ]:
optimal_frame = extract_spectra(
    image=science,
    tracemask=tracemask,
    fibermap=fibermap,
    method="optimal",
    flat_image=flat,
    aperture_radius=3,
    gain=1.0,
    read_noise=1.0,
)

print(f"shape : {optimal_frame.shape}")
print(f"meta  : {optimal_frame.meta}")

try:
    extract_spectra(
        image=science, tracemask=tracemask, fibermap=fibermap,
        method="optimal", flat_image=flat,
        variance=np.ones_like(science),
    )
except ValueError as error:
    print(f"\npassing variance to optimal is refused: {error}")

## 4. How the two compare

Start with the case that can be reasoned about exactly. One fiber, no
neighbours inside the aperture, and a noiseless flat. Then the flat carries the
profile exactly, the weighted sums cancel, and FOX must return the boxcar sum.

In [ ]:
n_fib, n_col, center, sigma, level = 500, 60, 30.0, 2.0, 2000.0
x = np.arange(n_col)
profile = np.exp(-0.5 * ((x - center) / sigma) ** 2)
spectrum = np.full(n_fib, 100.0)

flat_clean = level * profile[np.newaxis, :] * np.ones((n_fib, 1))
science_clean = spectrum[:, np.newaxis] * profile[np.newaxis, :]
trace_clean = np.full((1, n_fib), center)

box_clean, _, _ = extract_boxcar(science_clean, trace_clean, aperture_radius=3)
fox_clean, _, _ = extract_optimal(science_clean, flat_clean, trace_clean,
                                  aperture_radius=3)

print(f"noiseless flat, isolated fiber:")
print(f"  max relative difference : {np.max(np.abs(fox_clean - box_clean) / box_clean):.1e}")
print(f"  boxcar sum              : {box_clean[0, 0]:.6f}")

On the real 8 px spacing that is no longer true, and the reason is worth being
precise about. Two effects enter:

1. The aperture of one fiber reaches into its neighbours, so a boxcar sum
   collects part of their light.
2. The flat carries noise. Boxcar sums the *noisy* flat from row to row; FOX
   builds its weights from the smooth trend of the flat instead.

The second effect is the one visible below: the two spectra differ by the amount
of noise in the flat, not by a difference in what they measure. Expect the
median difference to sit near a percent, with the tail a few percent.

In [ ]:
difference = (optimal_frame.flux - boxcar_frame.flux) / boxcar_frame.flux
bright = boxcar_frame.flux > 1.0        # continuum and lines, away from the noise floor
relative = np.abs(difference[bright])

print(f"optimal versus boxcar on the 15-fiber frame:")
print(f"  rows compared              : {int(bright.sum())} of {bright.size}")
print(f"  median relative difference : {np.nanmedian(relative):.3%}")
print(f"  99th percentile            : {np.nanpercentile(relative, 99):.3%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
index = 7
axes[0].plot(boxcar_frame.wave, boxcar_frame.flux[index], lw=0.9, label="boxcar")
axes[0].plot(optimal_frame.wave, optimal_frame.flux[index], lw=0.9, label="optimal")
axes[0].set_xlabel("pixel (dispersion direction)")
axes[0].set_ylabel("flux [ADU]")
axes[0].set_title(f"Fiber {index}, both methods")
axes[0].legend()
axes[1].plot(boxcar_frame.wave[bright[index]],
             difference[index][bright[index]], lw=0.7)
axes[1].axhline(0, color="k", ls="--", lw=0.8)
axes[1].set_xlabel("pixel (dispersion direction)")
axes[1].set_ylabel("(optimal - boxcar) / boxcar")
axes[1].set_title("Relative difference")
plt.tight_layout()
plt.show()

## 5. What FOX buys

Weighting by the profile pays off when part of the aperture holds no light. The
pixels in the wings of the aperture carry read noise but almost no signal;
boxcar counts them all the same, FOX down-weights them.

Measure it: one isolated fiber, read noise large enough to dominate, and vary
the aperture radius. The profile has a 2 px sigma, which is the width measured
on the instrument.

In [ ]:
rng = np.random.default_rng(5)
read_noise = 30.0
science_e = spectrum[:, np.newaxis] * profile[np.newaxis, :]

print(f"{'radius':>7} {'boxcar':>9} {'optimal':>9} {'ratio':>7}")
for radius in (3, 4, 5, 6, 8):
    boxcar_scatter, optimal_scatter = [], []
    for _ in range(100):
        noisy = science_e + rng.normal(
            0.0, np.sqrt(np.clip(science_e, 0, None) + read_noise ** 2))
        noisy = np.clip(noisy, 0, None)
        box, _, _ = extract_boxcar(noisy, trace_clean, aperture_radius=radius)
        fox, _, _ = extract_optimal(noisy, flat_clean, trace_clean,
                                    aperture_radius=radius,
                                    gain=1.0, read_noise=read_noise)
        boxcar_scatter.append(box[0].std())
        optimal_scatter.append(fox[0].std())
    ratio = np.mean(optimal_scatter) / np.mean(boxcar_scatter)
    print(f"{radius:7d} {np.mean(boxcar_scatter):9.3f} {np.mean(optimal_scatter):9.3f} {ratio:7.3f}")

At the pipeline default of radius 3, the aperture already holds 92% of the
profile, so optimum weighting buys only about 3%. Widen the aperture to 8 px and
the gain grows to about 17%, because more of the aperture becomes empty pixels.

Two consequences for how you use the pipeline:

- **At this detector's noise level, FOX is not a large signal-to-noise win.**
  The gain register is 1 electron per ADU and the read noise is about 1
  electron, so real exposures are photon-noise limited. Repeating the
  comparison with `read_noise=1.0` on a bright signal puts the ratio at 1.000:
  the two methods are then statistically identical.
- **What FOX does buy is a spectrum that does not carry the flat's noise**, plus
  a graceful response to bad pixels and cosmic rays, because a bad pixel can be
  down-weighted instead of summed. That is a robustness argument, not a
  signal-to-noise one.

The choice is therefore: boxcar unless you have a reason. Use optimal when you
must widen the aperture, when the flat is noisy, or when you need the noise
model that FOX propagates into `ivar`.

## 6. The orchestrators

Two wrappers chain identification, tracing and extraction, so a script does not
have to.

- `run_quick_reduction` identifies on the flat and extracts with boxcar.
- `run_reduction` does the same but with optimal extraction by default.

In [ ]:
quick = run_quick_reduction(
    image=science, flat_image=flat,
    n_blocks_expected=3, n_fibers_per_block_expected=5,
    aperture_radius=3,
)
full = run_reduction(
    image=science, flat_image=flat,
    n_blocks_expected=3, n_fibers_per_block_expected=5,
    method="optimal", aperture_radius=3, gain=1.0, read_noise=1.0,
)

print(f"run_quick_reduction : {quick.shape}, method {quick.meta['METHOD']}")
print(f"run_reduction       : {full.shape}, method {full.meta['METHOD']}, "
      f"gain {full.meta['GAIN']}, read noise {full.meta['RDNOISE']}")

## 7. Save and load

`FiberFrame` writes to FITS, so a reduction step can be checkpointed and picked
up later.

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "spectra.fits")
    boxcar_frame.to_fits(path, overwrite=True)
    restored = type(boxcar_frame).from_fits(path)

    print(f"written to a temporary file: {os.path.basename(path)}")
    print(f"restored shape {restored.shape}")
    print(f"flux identical : {np.array_equal(restored.flux, boxcar_frame.flux)}")
    print(f"ivar identical : {np.array_equal(restored.ivar, boxcar_frame.ivar)}")
    print(f"mask identical : {np.array_equal(restored.mask, boxcar_frame.mask)}")

## Summary

| Step | Call | Result |
| --- | --- | --- |
| Boxcar | `extract_spectra(..., method="boxcar")` | aperture sum |
| Optimal | `extract_spectra(..., method="optimal", flat_image=..., gain=..., read_noise=...)` | flat-relative weighted sum |
| Both at once | `run_quick_reduction`, `run_reduction` | `FiberFrame` |
| Checkpoint | `FiberFrame.to_fits`, `FiberFrame.from_fits` | FITS round trip |

Measured on this frame:

- On an isolated fiber with a noiseless flat the two methods agree to machine
  precision, to 1 part in 10^16.
- On the 15-fiber frame with the instrument's spacing they differ by a median
  of 0.7%, with a 99th percentile of 3%. That difference is the noise of the
  flat, which boxcar sums row by row and FOX does not.
- When read noise dominates, optimal extraction lowers the scatter by 3% at the
  default aperture radius 3, and by 17% at radius 8. When photon noise dominates
  the two are identical.

**Where to go next:** tutorial 05 solves the pixel-to-wavelength mapping, which
is what turns the x-axis of these plots from pixels into Angstrom.